# CyberSentry — 05 · GenAI Explanations & Incident Reports (Stage 3)

**Goal:** turn the ML/DL detections into a clear explanation and a structured **incident report** for the security analyst.

**How it stays grounded (no made-up facts):**
1. **Code builds the evidence**: attack type, number of flows, confidence, which features are unusual compared with normal traffic, top ports
2. A **curated knowledge base** (written by us, in `src/genai_report.py`) adds the attack description, MITRE ATT&CK mapping, severity and standard actions
3. The **LLM only writes the text**, in a fixed JSON format (Groq *structured outputs*), with instructions to use only the evidence
4. A **grounding check** scans the LLM's answer for numbers that are not in the evidence and flags them
5. If the API is unavailable, a **template report** is produced from the same evidence, so the dashboard never breaks

**LLM:** Groq API, model `openai/gpt-oss-20b` (production model with strict structured-output support).

**Code used:** `src/pipeline.py` (runs Stage 1 → Stage 2) and `src/genai_report.py` (evidence, LLM call, grounding check, report).

## 0. One-time setup: Groq API key

1. Sign up at **console.groq.com** → *API Keys* → **Create API Key** → copy it (you only see it once)
2. In the project root (`Cybersentry\`) create a file named **`.env`** containing one line:
   ```
   GROQ_API_KEY=paste_your_key_here
   ```
3. Make sure `.env` is listed in `.gitignore`. **Never commit an API key to GitHub.**

## 1. Setup

In [1]:
import json
import os
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
from dotenv import load_dotenv
from IPython.display import Markdown, display

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
from pipeline import CyberSentryPipeline
import genai_report as gr

load_dotenv(ROOT / ".env")
HAS_KEY = bool(os.environ.get("GROQ_API_KEY"))
print("Groq API key found:", HAS_KEY, "" if HAS_KEY else "-> reports will use the template fallback")

GITIGNORE = ROOT / ".gitignore"
if GITIGNORE.exists() and ".env" not in GITIGNORE.read_text():
    print("WARNING: .env is NOT in .gitignore. Add it before committing!")

GENAI_DIR = ROOT / "models" / "genai"; GENAI_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR = ROOT / "reports" / "samples"; REPORT_DIR.mkdir(parents=True, exist_ok=True)
SEED = 42

Groq API key found: True 


## 2. Learning the Groq API: your first call

Every chat API works the same way:
- **`model`**: which LLM to use
- **`messages`**: the conversation, as a list of `{"role": ..., "content": ...}`
  - `system`: instructions about *how* to behave
  - `user`: the actual request
- **`temperature`**: randomness (0 = focused and repeatable, 1 = creative). We use low values for reports.
- The answer is in `response.choices[0].message.content`

This cell sends one small request so you can see it work.

In [2]:
if HAS_KEY:
    from groq import Groq
    client = Groq()   # reads GROQ_API_KEY from the environment
    t0 = time.time()
    response = client.chat.completions.create(
        model=gr.DEFAULT_MODEL,
        messages=[
            {"role": "system", "content": "You are a concise cybersecurity tutor."},
            {"role": "user", "content": "In two sentences, what is a port scan?"},
        ],
        temperature=0.2,
    )
    print(response.choices[0].message.content)
    print(f"\n({time.time() - t0:.1f}s, tokens used: {response.usage.total_tokens})")
else:
    print("No API key yet - add it to .env (section 0) and re-run this cell.")

A port scan is a method of probing a computer or network device to identify which of its network ports are open, closed, or filtered by sending packets to a range of port numbers and observing the responses. It is used by attackers to find exploitable services and by administrators to audit security posture.

(1.1s, tokens used: 249)


## 3. Normal-traffic baseline

To explain *why* something looks like an attack, we compare it with normal traffic. The median of every feature over **BENIGN training flows** is computed once and saved (the dashboard reuses it).

In [3]:
pipe = CyberSentryPipeline(ROOT)
train = pd.read_parquet(ROOT / "data" / "processed" / "train.parquet")
baseline = gr.benign_baseline(train, pipe.s2_features)
json.dump(baseline, open(GENAI_DIR / "benign_baseline.json", "w"), indent=2)
del train
print(f"Saved baseline for {len(baseline)} features -> {GENAI_DIR / 'benign_baseline.json'}")

Saved baseline for 32 features -> d:\Engineering\SEM 5\MINI PROJECT II\Cybersentry\models\genai\benign_baseline.json


## 4. Simulate an analyst upload

We take a sample from the **test set** (normal traffic plus some of each attack), hide the labels, and run the full pipeline exactly as the dashboard will.

In [4]:
test = pd.read_parquet(ROOT / "data" / "processed" / "test.parquet")
parts = [g.sample(min(len(g), 20_000 if lbl == "BENIGN" else 400), random_state=SEED)
         for lbl, g in test.groupby("Label")]
upload = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)
true_labels = upload.pop("Label").replace({"Web Attack - Brute Force": "Web Attack", "Web Attack - XSS": "Web Attack",
                                           "Web Attack - Sql Injection": "Web Attack"})
upload = upload.drop(columns=["binary_label"])

t0 = time.time()
results = pipe.predict(upload)
print(f"Analysed {len(results):,} flows in {time.time() - t0:.1f}s\n")

summary = results.groupby("attack_type").agg(flows=("attack_type", "size"),
                                             mean_confidence=("confidence", "mean"),
                                             needs_review=("needs_review", "sum")).sort_values("flows", ascending=False)
summary["correct_vs_true_label"] = results.assign(t=true_labels.values).groupby("attack_type").apply(
    lambda g: (g["t"] == g.name).mean(), include_groups=False)
summary.style.format({"mean_confidence": "{:.2%}", "correct_vs_true_label": "{:.2%}"})

Analysed 23,820 flows in 0.1s



,flows,mean_confidence,needs_review,correct_vs_true_label
attack_type,,,,
BENIGN,20018,60.57%,13,99.82%
DoS GoldenEye,409,99.26%,6,97.80%
DoS Hulk,405,99.33%,3,98.27%
FTP-Patator,405,99.84%,1,98.52%
PortScan,403,99.84%,1,99.26%
DoS Slowhttptest,402,98.88%,3,98.26%
DDoS,400,100.00%,0,100.00%
SSH-Patator,394,99.41%,2,99.75%
DoS slowloris,393,99.22%,2,99.75%


## 5. Generate one incident report per detected attack type

One LLM call per attack type (not per flow) keeps us well inside Groq's free-tier limits and is how an analyst actually wants it: one incident per kind of activity.

In [5]:
threats = [a for a in summary.index if a != "BENIGN"]
reports, checks = {}, []

for attack in threats:
    flows = results[results["attack_type"] == attack]
    evidence = gr.build_evidence(flows, attack, baseline, total_flows=len(results))
    md_report, narrative, check, source = gr.generate_report(evidence, use_llm=HAS_KEY)
    reports[attack] = md_report
    fname = REPORT_DIR / f"{attack.replace(' ', '_').replace('-', '')}.md"
    fname.write_text(md_report, encoding="utf-8")
    checks.append({"attack_type": attack, "source": source, "grounded": check["grounded"],
                   "unsupported_numbers": ", ".join(check["unsupported_numbers"]) or "-"})
    print(f"{attack:<18} -> {fname.name:<28} grounded={check['grounded']}")
    if HAS_KEY:
        time.sleep(3)   # stay well under the per-minute rate limit

pd.DataFrame(checks)

DoS GoldenEye      -> DoS_GoldenEye.md             grounded=True
DoS Hulk           -> DoS_Hulk.md                  grounded=True
FTP-Patator        -> FTPPatator.md                grounded=True
PortScan           -> PortScan.md                  grounded=True
DoS Slowhttptest   -> DoS_Slowhttptest.md          grounded=True
DDoS               -> DDoS.md                      grounded=True
SSH-Patator        -> SSHPatator.md                grounded=True
DoS slowloris      -> DoS_slowloris.md             grounded=True
Web Attack         -> Web_Attack.md                grounded=False
[genai] LLM failed, using template: Groq call failed after 3 attempts: Error code: 400 - {'error': {'message': "Failed to validate JSON. Please adjust your prompt. See 'failed_generation' for more details.", 'type': 'invalid_request_error', 'code': 'json_validate_failed', 'failed_generation': ''}}
Bot                -> Bot.md                       grounded=True
Infiltration       -> Infiltration.md             

,attack_type,source,grounded,unsupported_numbers
0,DoS GoldenEye,Groq openai/gpt-oss-20b,True,-
1,DoS Hulk,Groq openai/gpt-oss-20b,True,-
2,FTP-Patator,Groq openai/gpt-oss-20b,True,-
3,PortScan,Groq openai/gpt-oss-20b,True,-
4,DoS Slowhttptest,Groq openai/gpt-oss-20b,True,-
5,DDoS,Groq openai/gpt-oss-20b,True,-
6,SSH-Patator,Groq openai/gpt-oss-20b,True,-
7,DoS slowloris,Groq openai/gpt-oss-20b,True,-
8,Web Attack,Groq openai/gpt-oss-20b,False,90
9,Bot,template (no LLM),True,-


## 6. Example report

In [6]:
example = "PortScan" if "PortScan" in reports else threats[0]
display(Markdown(reports[example]))

# Port Scan Detected: 403 Flows
**Incident ID:** CS-20261022-PORTSCAN-403  |  **Generated:** 2026-10-22 22:54  |  **Severity:** Medium  |  **Attack type:** PortScan

## Summary
A reconnaissance port scan was detected involving 403 flows, representing 1.69% of the analyzed traffic. The detection confidence is 99.84% with a minimum confidence of 46.72%.

## Detection details
- Flows detected: **403** (1.69% of analysed traffic)
- Model confidence: mean **99.84%**, minimum 46.72%
- Flows needing analyst review (confidence below threshold): 1
- MITRE ATT&CK: T1046 Network Service Discovery
- Top destination ports: 1216 (5 flows), 8180 (3 flows), 31337 (3 flows)

## What happened
The system identified a rapid probing of multiple ports, with unusually high packets per second and extremely short flow durations, indicating a scanning activity.

## Key evidence
- 403 flows detected, 1.69% of analyzed traffic.
- Mean confidence 99.84%, minimum confidence 46.72%.
- Flow Duration median 49 microseconds vs 46290 microseconds (0.00106 times normal).
- Bwd Packets/s median 20408 packets per second vs 23.87 packets per second (855.0 times normal).
- Top destination ports: 1216 (5 flows), 8180 (3 flows), 31337 (3 flows).

| Feature | Unit | Attack median | Normal median | x normal |
|---|---|---:|---:|---:|
| Flow Duration | microseconds | 49 | 46290 | 0.00106 |
| Bwd Packets/s | packets per second | 20408 | 23.87 | 855.0 |
| Flow IAT Max | microseconds | 49 | 32075 | 0.00153 |
| Flow Packets/s | packets per second | 40816 | 84.5 | 483.0 |
| Flow IAT Std | microseconds | 0 | 436 | 0.0 |
| Flow IAT Mean | microseconds | 49 | 16213 | 0.00302 |

## Potential impact
If the scanning source is malicious, it could lead to discovery of open services and potential exploitation.

## Recommended actions
1. Identify and monitor the source of the scanning activity.
2. Close or firewall any unnecessary open ports that were targeted.
3. Investigate for any follow‑up attacks originating from the same source.

## Analyst note
Verify the source of the scans (IP addresses are not available in the current data) and confirm that the identified ports are secured. This report is AI‑assisted and requires analyst verification due to one flow needing review.

---
*Narrative source: Groq openai/gpt-oss-20b. Grounding check: all numbers match the evidence. CyberSentry assists analysts; it does not replace them.*

## 7. Does the grounding check actually catch made-up numbers?

We take a real narrative and deliberately insert a fake number, to show the check flags it. This demonstrates the hallucination safeguard for the report.

In [7]:
attack = threats[0]
evidence = gr.build_evidence(results[results["attack_type"] == attack], attack, baseline, total_flows=len(results))
narrative = gr.template_narrative(evidence)
print("Clean narrative       ->", gr.grounding_check(narrative, evidence))

tampered = dict(narrative)
tampered["summary"] = narrative["summary"] + " The attacker sent 987654 packets from 10.0.0.66."
print("With invented numbers ->", gr.grounding_check(tampered, evidence))

Clean narrative       -> {'grounded': True, 'unsupported_numbers': []}
With invented numbers -> {'grounded': False, 'unsupported_numbers': ['10.0.0.66', '987654']}


## 8. Summary

*(Fill in from the outputs above.)*

- **Pipeline on simulated upload (section 4):** flows analysed, attack types detected, accuracy vs. hidden labels
- **Reports generated (section 5):** one per detected attack type, saved in `reports/samples/`
- **Grounding (section 5 + 7):** how many reports passed the check; section 7 shows the check catches invented numbers
- **Design choices that limit hallucination:** evidence built by code, curated knowledge base, strict JSON output, low temperature, numeric grounding check, template fallback

Next: the Streamlit dashboard (upload CSV → detection → reports), reusing `src/pipeline.py` and `src/genai_report.py`.